<a href="https://colab.research.google.com/github/dev-105-data/IBM-QC-Today/blob/main/From_Zero_to_Quantum_Running_Your_First_Program.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantum Lab: Build and Run Your First Quantum Program
### *Course: "Use a Quantum Computer Today" | Episode 1*

---

**Author:** Carlos Araque  
**Title:** IBM Qiskit Advocate  
**Compiled Using:** Gemini AI (Adaptive Collaborator)

**About this Notebook:** This notebook is specifically designed for **Lesson 1** of the IBM Qiskit course. It provides the hands-on environment to execute the "Hello World" of quantum computing—the Bell State—on both a simulator and real IBM hardware.

**Course Roadmap:**
* **Lesson 1 (This Notebook):** Build and run your first quantum program. [Video Link](https://ibm.co/6042EKOii)
* **Lesson 2:** Quantum mechanics basics (Superposition & Entanglement). [Video Link](https://ibm.co/6043EKOic)
* **Lesson 3:** Your first quantum experiment. [Video Link](https://ibm.co/6044EKOiY)

**Project Repository:** [GitHub - CarlosAraque/Quantum Computer Today](https://github.com/dev-105-data/IBM-QC-Today)

---

**Part 1: Environment Setup and Validation**  
**Cell 1: Installation**  
First, we install the core Qiskit packages. This includes qiskit for circuit building, qiskit-aer for local simulation, and qiskit-ibm-runtime to access real hardware.

In [ ]:
# Install Qiskit and related tools
!pip install qiskit qiskit-aer qiskit-ibm-runtime matplotlib pylatexenc

**Cell 2: Dependency Validation (The "Seat Belt" Step)**  
Following the video's advice, we perform an environment check.  This code confirms the installed versions and ensures that the drawing engine is working.

In [ ]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime
import matplotlib

# Check and print versions
print("--- Dependency Validation ---")
print(f"Qiskit version: {qiskit.__version__}")
print(f"Aer version: {qiskit_aer.__version__}")
print(f"IBM Runtime version: {qiskit_ibm_runtime.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")

# Verify if the visualization works
from qiskit import QuantumCircuit
test_qc = QuantumCircuit(1)
test_qc.h(0)
print("\nVisualization Check:")
display(test_qc.draw('mpl'))
print("Setup successful!")

**Part 2: Building the Quantum Circuit**  
**Cell 3: Imports and Initialization**  
We import the classes needed to define qubits, operations, and the tools to visualize the measurement results.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

# Define the Bell State circuit (Phi Plus)
qc = QuantumCircuit(2)

# Apply a Hadamard gate to qubit 0 (creates superposition)
qc.h(0)

# Apply a CNOT gate (Control: 0, Target: 1) to create entanglement
qc.cx(0, 1)

# Map the quantum state to classical bits
qc.measure_all()

# Draw the final circuit
print("Circuit for Phi Plus Bell State:")
qc.draw('mpl')

**Part 3: Execution Logic**  
**Cell 4: Helper Function for Execution**  
This function handles the Transpilation (optimizing the code for the specific hardware) and the Execution via the Sampler primitive.

In [ ]:
def run_quantum_circuit(circuit, backend, shots=1024):
    """
    Optimizes and executes a circuit on the specified backend.
    """
    # Create the Pass Manager for optimization
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)

    # Transform the circuit into an Instruction Set Architecture (ISA) circuit
    isa_circuit = pm.run(circuit)

    # Initialize the Sampler and run the job
    sampler = Sampler(mode=backend)
    job = sampler.run([isa_circuit], shots=shots)

    # Retrieve result and extract counts
    result = job.result()
    return result[0].data.meas.get_counts()

**Cell 5: Running on the Local Simulator**  
Before using real hardware (which costs time and credits), we run it on the AerSimulator to get the "perfect" theoretical result.

In [ ]:
from qiskit_aer import AerSimulator

# Select the simulator as our backend
sim_backend = AerSimulator()

# Run the circuit
counts = run_quantum_circuit(qc, sim_backend)

print(f"Simulation Counts: {counts}")
plot_histogram(counts, title="Ideal Bell State Results (Simulator)")

**Important Notes:**  
**Superposition:** Created by the **H** (Hadamard) gate.

**Entanglement:** Created by the **CX** (CNOT) gate. In a Bell State, measuring one qubit instantly tells you the state of the other.

**Noisy Hardware:** If you eventually run this on a real IBM computer, you will see small bars for **01** and **10** due to environmental interference, unlike the "clean" results of the simulator.

**Cell 6: Connecting to IBM Quantum (Real Hardware)**  
This section retrieves your API Token from the IBM Cloud "my-token"


In [ ]:
QiskitRuntimeService.save_account(channel='ibm_quantum_platform', token='my-token', overwrite=True, set_as_default=True)
service = QiskitRuntimeService(channel='ibm_quantum_platform')

# Load saved credentials
service = QiskitRuntimeService()

# Use the least busy backend, or uncomment the loading of a specific backend like "ibm_fez".
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=127)
# backend = service.backend("ibm_fez")
print(backend.name)

**Cell 7: Running on Real Hardware**  
Now we send the circuit to the actual processor in the lab. Note that your job might wait in a queue for a few minutes.

In [ ]:
# Use the helper function defined earlier to run on the real backend
print(f"Submitting job to {backend.name}. Please wait...")

# We use 1024 shots to get clear statistics
real_counts = run_quantum_circuit(qc, backend, shots=1024)

# Plot the results from the real computer
print(f"Real Hardware Results: {real_counts}")
plot_histogram(real_counts, title=f"Real Hardware Results from {backend.name}")

**Understanding the Hardware Results**  
When you compare the Simulator vs. Real Hardware results, you will notice a key difference mentioned in the video:

**Ideal (Simulator):** You only see 00 and 11.

**Real (Hardware):** You will see small occurrences of 01 and 10. This is Quantum Noise. Because the qubits are physical objects (superconducting circuits), they can be affected by heat or magnetic interference, leading to slight measurement errors.

**Summary of your Notebook Structure:**
Validation: Checking versions (Seat belt step).

Construction: Defining the Bell State (Entanglement).

Local Test: Running the **AerSimulator** (Perfect world).

Cloud Connection: Using Colab Secrets to reach IBM.

Quantum Run: Executing on a 127-qubit machine (Real world).

Congratulations! You now have a complete, professional-grade Quantum Computing Notebook ready for Google Colab.

**Bonus Track by CA**
**4 Estados de Bell**

**Cell 1: Installation**

In [ ]:
# Install necessary libraries
!pip install qiskit qiskit-aer matplotlib pylatexenc

**Cell 2: Creating the 4 Bell States (Code & Documentation)**  
Copy this into a new cell. This script creates the four different entangled states and runs them on the **AerSimulator**.

In [ ]:
import qiskit
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler

def create_bell_state(state_type):
    """
    Creates one of the four Bell States.
    Phi+ : (|00> + |11>) / sqrt(2)
    Phi- : (|00> - |11>) / sqrt(2)
    Psi+ : (|01> + |10>) / sqrt(2)
    Psi- : (|01> - |10>) / sqrt(2)
    """
    qc = QuantumCircuit(2)

    if state_type == "phi_plus":
        # Standard Bell State (The one from the video)
        qc.h(0)
        qc.cx(0, 1)

    elif state_type == "phi_minus":
        # Apply X gate to change phase
        qc.x(0)
        qc.h(0)
        qc.cx(0, 1)

    elif state_type == "psi_plus":
        # Entangled but opposite states
        qc.x(1)
        qc.h(0)
        qc.cx(0, 1)

    elif state_type == "psi_minus":
        # Opposite states with phase difference
        qc.x(1)
        qc.x(0)
        qc.h(0)
        qc.cx(0, 1)

    qc.measure_all()
    return qc

# Initialize Simulator
backend = AerSimulator()
sampler = Sampler(mode=backend)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)

# List of states to test
states = ["phi_plus", "phi_minus", "psi_plus", "psi_minus"]

print("--- Running Bell State Simulations ---")

for state in states:
    print(f"\nProcessing: {state}")
    # 1. Create Circuit
    circuit = create_bell_state(state)

    # 2. Transpile for the simulator
    isa_circuit = pm.run(circuit)

    # 3. Execute
    result = sampler.run([isa_circuit], shots=1024).result()
    counts = result[0].data.meas.get_counts()

    print(f"Results for {state}: {counts}")
    # Display the circuit for each state
    display(circuit.draw('mpl'))

**Quick Documentation (English)**  
1. Phi Plus ($\Phi^+$)Logic: Start with $|00\rangle$, apply $H$ to qubit 0, then $CNOT$ (0 to 1).Result: You will see 50% 00 and 50% 11. The qubits always agree.

2. Phi Minus ($\Phi^-$)Logic: Start with $|1\rangle$ on qubit 0 (using an $X$ gate), then apply $H$ and $CNOT$.Result: Statistically looks like $\Phi^+$, but there is a phase difference (important for interference).

3. Psi Plus ($\Psi^+$)Logic: Start with $|01\rangle$ (apply $X$ to qubit 1), then $H$ and $CNOT$.Result: You will see 50% 01 and 50% 10. The qubits always disagree.

4. Psi Minus ($\Psi^-$)Logic: Both qubits start at different states before the entanglement.Result: Similar to $\Psi^+$, the qubits will always be different (01 or 10).

**IBM Quantum Network Connection via Google Colab Secrets**

In [ ]:
# ======================================================================================
# IBM Quantum Network Connection via Google Colab Secrets
# --------------------------------------------------------------------------------------
# This cell establishes a connection to the IBM Quantum Platform using
# Google Colab's native secret management (the key icon 🔑 on the left sidebar).
#
# Pre-requisite:
# 1. Open the "Secrets" tab.
# 2. Add a key named 'IBM_QUANTUM_TOKEN' and paste your API token.
# 3. Grant notebook access to this secret.
# ======================================================================================

from google.colab import userdata
from qiskit_ibm_runtime import QiskitRuntimeService
import logging

# Configure basic logging for transparency
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def connect_to_quantum_runtime():
    """
    Retrieves credentials from Colab Secrets and initializes
    the Qiskit Runtime Service.
    """
    try:
        # Retrieve the secret token
        ibm_token = userdata.get('IBM_QUANTUM_TOKEN')

        # Save and overwrite local credentials for this session
        QiskitRuntimeService.save_account(
            channel='ibm_quantum_platform',
            token=ibm_token,
            overwrite=True,
            set_as_default=True
        )

        # Initialize the service
        service = QiskitRuntimeService()
        logger.info("Successfully authenticated with IBM Quantum.")
        return service

    except userdata.SecretNotFoundError:
        logger.error("Secret 'IBM_QUANTUM_TOKEN' not found. Please check your Secrets tab.")
        return None
    except Exception as e:
        logger.error(f"An unexpected error occurred: {e}")
        return None

# --- Execution ---
service = connect_to_quantum_runtime()

if service:
    # Query for the least busy hardware backend (Eagle/Heron processors)
    # min_num_qubits=127 ensures we target Utility-scale systems
    print("\nFetching the least busy Utility-scale hardware...")
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=127)

    print("-" * 50)
    print(f"ACTIVE BACKEND: {backend.name}")
    print(f"QUBITS:         {backend.num_qubits}")
    print(f"STATUS:         {backend.status().status_msg}")
    print("-" * 50)

---
## 🏁 Conclusion & What's Next

Congratulations! You have successfully built, validated, and executed an entangled quantum circuit. Whether you ran it on the **AerSimulator** or took the leap into **Real IBM Quantum Hardware**, you've officially taken your first step from classical to quantum.

### 🔜 Looking Ahead
In the next installment, we will dive deeper into the "why" behind the "how." We'll explore:
* **The Mathematics of Superposition:** Moving beyond the buzzwords.
* **Phase and Interference:** Understanding why the Bell States differ.
* **Advanced Circuit Building:** Preparing for real-world applications.

**Keep experimenting, stay curious, and I'll see you in the next notebook!**

---
**Carlos Araque** *Data Engineer | Qiskit Advocate*